In [2]:
!pip install google-genai

   ---------------------------------------- 0.0/950.8 kB ? eta -:--:--
   ---------------------------------------- 950.8/950.8 kB 8.5 MB/s  0:00:00

   ---------------------------------------- 0/6 [websockets]
   ---------------------------------------- 0/6 [websockets]
   ---------------------------------------- 0/6 [websockets]
   ---------------------------------------- 0/6 [websockets]
   ------ --------------------------------- 1/6 [tenacity]
   ------------- -------------------------- 2/6 [pyasn1]
   ------------- -------------------------- 2/6 [pyasn1]
   ------------- -------------------------- 2/6 [pyasn1]
   -------------------- ------------------- 3/6 [pyasn1-modules]
   -------------------- ------------------- 3/6 [pyasn1-modules]
   -------------------- ------------------- 3/6 [pyasn1-modules]
   -------------------- ------------------- 3/6 [pyasn1-modules]
   -------------------- ------------------- 3/6 [pyasn1-modules]
   -------------------- ------------------- 3/6 [pya

In [ ]:
import sys
import asyncio
from google import genai  # 💡 เปลี่ยนมาใช้ไลบรารีของ Google Gemini

# บังคับใช้ Event Loop สำหรับ Windows
if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

from mcp.client.stdio import stdio_client
from mcp import StdioServerParameters
from mcp.client.session import ClientSession

async def run_ai_poc():
    sales_data_json = None
    
    # ==========================================
    # ส่วนที่ 1: เชื่อมต่อ MCP เพื่อดึงข้อมูล (โค้ดเดิมที่ทำงานสมบูรณ์แล้ว)
    # ==========================================
    print("⏳ กำลังเชื่อมต่อกับ MCP Server...")
    server_params = StdioServerParameters(
        command=sys.executable, 
        args=["mcp_server.py"]
    )

    with open("mcp_server_errors.log", "w", encoding="utf-8") as err_file:
        try:
            async with stdio_client(server_params, errlog=err_file) as (read, write):
                async with ClientSession(read, write) as session:
                    await session.initialize()
                    print("✅ เชื่อมต่อ MCP Server สำเร็จ!\n")
                    
                    print("📥 กำลังให้ MCP ดึงข้อมูลจาก REST API...")
                    result = await session.call_tool("get_sales_data", arguments={})
                    sales_data_json = result.content[0].text
                    print("📦 ดึงข้อมูลสำเร็จ และปิดการเชื่อมต่อ MCP แล้ว\n")
                    
        except Exception as e:
            print(f"❌ เกิดข้อผิดพลาดฝั่ง MCP: {type(e).__name__} - {str(e)}")
            return

    # ==========================================
    # ส่วนที่ 2: ส่งข้อมูลให้ Gemini วิเคราะห์
    # ==========================================
    if sales_data_json:
        prompt = f"""
        คุณคือนักวิเคราะห์ข้อมูลมืออาชีพ นี่คือข้อมูลยอดขายในรูปแบบ JSON:
        {sales_data_json}
        
        ช่วยวิเคราะห์และสรุปให้หน่อยว่า:
        1. สินค้าหมวดหมู่ไหน (category) ทำยอดขายรวม (total_amount) ได้มากที่สุด?
        2. ใครคือลูกค้ารายใหญ่ที่สุด (ยอดซื้อรวมสูงสุด)?
        """
        
        print("🤖 เตรียมส่งข้อมูลให้ AI วิเคราะห์...")
        print("==========================================")
        print("🤖 กำลังส่ง Prompt ให้ Gemini วิเคราะห์ข้อมูลจริง...")
        
        try:
            # ⚠️ นำ API Key ของ Gemini มาใส่ตรงนี้ครับ
            client = genai.Client(api_key="put your_gemini_api_key_here")

            # ใช้โมเดล gemini-2.5-flash ซึ่งวิเคราะห์ JSON ได้ไวและแม่นยำมาก
            response = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=prompt,
            )

            print("\n💡 คำตอบวิเคราะห์จาก Gemini (ผ่านระบบ MCP):")
            print("------------------------------------------")
            print(response.text)
            print("------------------------------------------")
            
        except Exception as e:
            print(f"\n❌ เกิดข้อผิดพลาดฝั่ง Gemini: {type(e).__name__}")
            print(f"📝 รายละเอียด: {str(e)}")

# สั่งรันฟังก์ชัน
await run_ai_poc()

⏳ กำลังเชื่อมต่อกับ MCP Server...
✅ เชื่อมต่อ MCP Server สำเร็จ!

📥 กำลังให้ MCP ดึงข้อมูลจาก REST API...
📦 ดึงข้อมูลสำเร็จ และปิดการเชื่อมต่อ MCP แล้ว

🤖 เตรียมส่งข้อมูลให้ AI วิเคราะห์...
🤖 กำลังส่ง Prompt ให้ Gemini วิเคราะห์ข้อมูลจริง...

💡 คำตอบวิเคราะห์จาก Gemini (ผ่านระบบ MCP):
------------------------------------------
ในฐานะนักวิเคราะห์ข้อมูลมืออาชีพ ผมได้ทำการวิเคราะห์ข้อมูลยอดขายที่ให้มา และขอสรุปผลดังนี้ครับ:

---

### **การวิเคราะห์ยอดขาย**

**1. สินค้าหมวดหมู่ไหน (category) ทำยอดขายรวม (total_amount) ได้มากที่สุด?**

จากการรวบรวมยอดขายรวมตามหมวดหมู่สินค้า พบว่า:

*   **Smartphone:** 117,700 บาท (iPhone 15 Pro, Samsung Galaxy S24)
*   **Laptop:** 69,800 บาท (MacBook Air, Dell Inspiron 15)
*   **Tablet:** 71,700 บาท (iPad Air)
*   **Accessories:** 42,500 บาท (AirPods Pro, Sony WH-1000XM5, Logitech MX Master 3S)
*   **Smartwatch:** 15,900 บาท (Apple Watch Series 9)
*   **Monitor:** 7,900 บาท (Asus Gaming Monitor)

**สรุป:** **หมวดหมู่ "Smartphone" ทำยอดขายรวมได้มากที่สุด ด้ว